# Notebook 03 - Treinamento e Selecao de Modelo

## Contexto

Este notebook seleciona o modelo e a estrategia de tratamento de desbalanceamento
para o diagnostico de hipertireoidismo. A selecao e feita exclusivamente por
cross-validation sobre o conjunto de treino. O conjunto de teste NAO e carregado
nem tocado neste notebook; ele fica reservado para a avaliacao final no notebook 04.

## Ponto de partida

- Fonte: artefato do notebook 02 (`X_train.csv`, `y_train.csv`), no estado pos-deterministico e pre-estatistico.
- X_train chega com 18 colunas [ajustado para 17 apos analise de interferencia e leakege clinico na execução do notebook], `referral source` ainda como texto e NaN preservados em `age` e nos labs.
- As operacoes estatisticas (OneHotEncoder, RobustScaler, KNNImputer) foram deslocadas do notebook 02 para o Pipeline deste notebook. Motivo: elas aprendem parametros de conjunto e, se aplicadas antes do cross-validation, vazam informacao entre folds. Dentro do Pipeline elas dao fit apenas na porcao de treino de cada fold.

## Objetivo

Comparar modelos e estrategias de balanceamento por cross-validation estratificada
e escolher a combinacao vencedora com base nas metricas definidas, sem usar o
teste para nenhuma decisao.

- Modelos: LogisticRegression (baseline), RandomForest, XGBoost.
- Estrategias de balanceamento: sem correcao (raw), class_weight, SMOTE.
- Metricas prioritarias: Recall, F1-score, AUC-ROC. Recall e critico pelo custo clinico de falso negativo; AUC-ROC avalia a separacao das classes independente de limiar.

## Desbalanceamento

Classe positiva em 92.29% do treino. Um classificador que preveja sempre positivo acerta 92% e ainda assim e inutil, porque erra toda a classe negativa. Por isso a acuracia e ignorada e a leitura se concentra no comportamento da classe minoritaria.

## Disciplina de validacao

- Toda comparacao roda em `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
- O `random_state` fica fixo em todas as rodadas para que as comparacoes entre modelos sejam justas (mesmos folds).
- SMOTE e aplicado dentro do Pipeline (`imblearn`), logo reamostra apenas o treino de cada fold, nunca a particao de validacao.

## Indice

1. Carga do artefato (X_train, y_train)
2. Definicao do Pipeline de preprocessing (ColumnTransformer + KNNImputer)
3. Definicao do cross-validation e das metricas
4. Experimentos
   - 4.1 Baseline: LogisticRegression em raw, class_weight e SMOTE
   - 4.2 Modelos de arvore, conforme leitura do baseline
5. Consolidacao das metricas de CV e escolha do vencedor

## Referencia

- Notebook 01 (EDA) e Notebook 02 (preprocessing): decisoes de feature e do artefato de preprossamento.

---

**Fronteira do projeto:** este notebook termina na escolha do vencedor. O teste
so aparece no notebook 04, treinando o Pipeline vencedor uma unica vez no treino
completo e medindo no teste, sem retorno para ajuste.

In [21]:
import pandas as pd # manipulação dos dados
import numpy as np # suporte de calculos
from sklearn.model_selection import StratifiedKFold, cross_validate # validação cruzada e estratificação
from imblearn.pipeline import Pipeline # pipeline
from sklearn.compose import ColumnTransformer #preprossing de pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler # codificação da categórica e escala das contínuas
from sklearn.impute import KNNImputer # imputação por vizinhos (KNN)
from imblearn.over_sampling import SMOTE # balancemento
from sklearn.linear_model import LogisticRegression # modelo
from sklearn.ensemble import RandomForestClassifier # modelo
from xgboost import XGBClassifier # modelo
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, classification_report # metricas de avaliação
from sklearn.model_selection import cross_val_predict # metricas de avaliação de modelo

In [2]:
# carregamento da base
X_train = pd.read_csv('X_train.csv',  sep=',')
y_train = pd.read_csv('y_train.csv', sep=',').squeeze('columns')

In [3]:
# validacao do estado do artefato antes do export
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'referral source dtype: {X_train['referral source'].dtype}')      # esperado: object
print(f'NaN em X_train: {X_train.isna().sum().sum()}')                   # esperado: > 0
print(f'NaN em y_train: {y_train.isna().sum().sum()}')                   # esperado: = 0
print(f'y_tarin: {y_train.value_counts(normalize=True)}')                # valro esperado ~92/8

X_train: (2829, 17) | y_train: (2829,)
referral source dtype: object
NaN em X_train: 1569
NaN em y_train: 0
y_tarin: binaryClass
1    0.922941
0    0.077059
Name: proportion, dtype: float64


In [4]:
# verifica inspecao dos nomes da colunas por meio do Dtypes
display(X_train.dtypes)

,0
age,float64
sex,float64
on antithyroid medication,int64
sick,int64
thyroid surgery,int64
query hypothyroid,int64
query hyperthyroid,int64
TSH,float64
T3 measured,int64
T3,float64


## Investigacao de leakage no baseline

O primeiro modelo analise para criação da baseline apresentou caracteristicas de risco que indivam alta chance de leakege, e risco de invalidação de todos os modelos posteriores, assim decidi realizar esta investigação

### Motivacao

O baseline (LogisticRegression, sem correcao de desbalanceamento) apresentou
metricas de validacao proximas do perfeito ja sem tratamento algum:

- F1: 0.987
- Recall: 0.993
- ROC-AUC: 0.991

Treino e validacao praticamente identicos, descartando overfitting. Um desempenho tao alto em baseline raw, num problema clinico desbalanceado (92/8), levantou a suspeita de leakage: alguma feature entregando a resposta ao modelo.

### Metodo

Inspecao dos coeficientes da LogisticRegression treinada no X_train completo
(apenas para diagnostico, sem valor de metrica). Coeficiente com valor absoluto
alto indica feature dominante na decisao.

### Achados dos coeficientes (dataset com todas as features)

| Feature | Coeficiente | Leitura |
|---------|-------------|---------|
| on thyroxine | [4.68] | Maior peso. Suspeito de leakage. |
| TSH | [-3.79] | Sinal clinico legitimo (TSH baixo, hipertireoidismo). |
| thyroid surgery | [3.03] | Peso alto, natureza ambigua. |
| flags measured (T3, T4U, FTI) | [0.008 a 0.047] | Peso desprezivel. Hipotese inicial descartada. |

Observacao: as flags de coleta, suspeitas iniciais, mostraram peso irrelevante.
A remocao delas nao se justificava. Investigar antes de remover evitou uma decisao errada.

## Verificacao empirica: remocao de on thyroxine

### Fundamento

A inspecao dos coeficientes da LogisticRegression (treinada no X_train completo,
apenas para diagnostico) revelou a seguinte distribuicao de peso:

- on thyroxine: coef 4.68 (maior peso absoluto da tabela)
- TSH: coef -3.79 (segundo maior, sinal clinico legitimo)
- thyroid surgery: coef 3.03 (terceiro, ambiguo, avaliado em etapa posterior)

As flags de coleta (T3 measured, T4U measured, FTI measured) apresentaram os
menores coeficientes da tabela (0.008 a 0.047), o que descartou a hipotese inicial
de que seriam a fonte do recall elevado.

### Diagnostico

on thyroxine é o maior contribuinte para a decisao do modelo. Clinicamente, é
tratamento para hipotireoidismo, condicao oposta ao alvo `hipertireoidismo`. Um
paciente sob tiroxina tende a nao ser caso positivo, e o modelo aprendeu esse
atalho em vez de aprender o quadro clinico do hipertireoidismo. Caracteriza
leakage: a natureza é estabelecida pela clinica, nao pelo numero.

### Experimento

Remover on thyroxine do conjunto de treino e re-executar o baseline (LogisticRegression
sem correcao de desbalanceamento), mantendo todo o restante identico (mesmo Pipeline,
mesmo StratifiedKFold, mesmas metricas). Altera-se uma unica variavel por vez para
isolar o efeito.

Leitura esperada:

- Queda moderada do recall (para patamar realista, mas ainda alto): confirma que
  on thyroxine inflava o resultado via leakage e que ha sinal legitimo sob ela.
- Queda acentuada: o modelo dependia excessivamente da feature, sinal residual fraco.

O numero medirá o tamanho do efeito da remocao, nao a natureza do leakage, já
estabelecida acima pelo raciocinio clinico.

### Experimento 1: remocao de on thyroxine

Fundamento clinico: on thyroxine e tratamento para hipotireoidismo, condicao oposta
ao alvo. O modelo aprendeu o atalho do tratamento em vez do quadro clinico. Leakage
estabelecido pela clinica, nao pelo numero.

Resultado da remocao (comparado ao baseline):

- Recall: [0.9935] para [0.9897]
- F1: [0.9874] para [0.9823]
- ROC-AUC: [0.9906] para [0.9857]

Queda minima. Conclusao: on thyroxine era leakage, porem redundante. A informacao dela tambem esta presente em outras features, e o modelo reconstroi quase a mesma separacao sem ela.

### Experimento 2: remocao de thyroid surgery (sobre o dataset ja sem thyroxine)

Analise clinica: cirurgia de tireoide eleva o TSH (confirmado no EDA). Pela fisiologia, TSH alto apontaria para a classe negativa, mas o coeficiente e positivo. A explicacao coerente e que o modelo capta o perfil de quem
opera (paciente tireoidiano, correlacionado com a classe positiva), nao o efeito
fisiologico, que o proprio TSH ja captura em separado.

Resultado da remocao (comparado ao dataset sem thyroxine):

- Recall: [0.9897] para [0.9893]
- F1: [0.9823] para [0.9815]
- ROC-AUC: [0.9857] para [0.9848]

Queda desprezivel novamente.

### Conclusao da investigacao

1. on thyroxine: leakage clinico confirmado. Removida do dataset no notebook 02.
2. thyroid surgery: sinal legitimo de perfil clinico, nao leakage. Mantida.
3. O desempenho quase perfeito nao vem de leakage dominante, e sim da alta
   separabilidade intrinseca do problema. Marcadores laboratoriais fortes (TSH)
   tornam as classes faceis de separar. A remocao das duas features de maior peso
   quase nao afetou as metricas, o que confirma que o sinal e redundante e distribuido.
4. Licao de metodo: coeficiente alto indica uso, nao dependencia critica. Nenhuma
   feature isolada sustenta o desempenho.

## Nota sobre a investigacao de leakage

As celulas de codigo da investigacao de leakage (extracao e analise dos coeficientes
da LogisticRegression, e os experimentos de remocao de features) foram removidas
deste notebook apos a conclusao da analise.

Motivo da remocao: essas celulas eram instrumentos de diagnostico, executados sobre uma versao anterior do dataset que ainda continha a feature `on thyroxine`. Apos a decisao de remover essa feature na origem (notebook 02), o dataset mudou, e reexecutar essas celulas geraria erro ou resultado inconsistente com o dataset atual. Manter codigo que nao roda de cima a baixo quebraria a reprodutibilidade do notebook.

Resumo do que foi investigado e concluido:

- Suspeita: recall proximo do perfeito no baseline cru, possivel leakage.
- Metodo: inspecao de coeficientes e remocao controlada de features suspeitas.
- Conclusao: `on thyroxine` era leakage clinico (removida na origem); `thyroid surgery`  e sinal legitimo (mantida); o desempenho alto decorre da alta separabilidade do problema, nao de leakage dominante.

A partir deste ponto, todos os resultados referem-se ao dataset final, sem `on thyroxine`.

In [5]:
# construção das variaveis continuas
continuas = ['age','TSH','T3','T4U','FTI']

# estrutura da inical para a pipeline

preprocessamento = ColumnTransformer(
    transformers=[
        ('scaler', RobustScaler(), continuas),
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['referral source'])
    ],
    remainder='passthrough'
)

In [6]:
# contrução apra  pipeline

data_processing = Pipeline(steps=[
    ('columntrans', preprocessamento),
    ('imputer', KNNImputer(n_neighbors=5,weights='distance', metric='nan_euclidean')),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(random_state=42))
])


In [7]:
#define função do smote
def contrucao_pipeline(modelo, usa_smote=False):
  passos = [  # passos fixos, iguais nos tres ramos
    ('columntrans', preprocessamento), # trata as colunas: escala continuas, codifica referral source
    ('imputer', KNNImputer(n_neighbors=5,weights='distance', metric='nan_euclidean')), # preenche NaN pelos 5 vizinhos, peso por distancia
  ]
  if usa_smote: # so entra quando usa_smote for True
    passos.append(('smote', SMOTE(random_state=42))) # reamostra a classe minoritaria, so no treino de cada fold
  passos.append(('model', modelo))
  return Pipeline(steps=passos) # devolve o Pipeline montado, pronto para o cross-validation

In [8]:
# preenchimento dos modelos
pipe_raw = contrucao_pipeline(modelo=LogisticRegression(random_state=42), usa_smote=False)
pipe_classweight = contrucao_pipeline(modelo=LogisticRegression(class_weight='balanced',random_state=42), usa_smote=False)
pipe_smote = contrucao_pipeline(modelo=LogisticRegression(random_state=42), usa_smote=True)

In [9]:
# construção do stratifielkfold
stratify = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
# chamado do CV(crossvalidadtion)
resultado_raw = cross_validate(
    estimator = pipe_raw,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

In [11]:
resultado_raw.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'train_accuracy', 'test_f1', 'train_f1', 'test_recall', 'train_recall', 'test_roc_auc', 'train_roc_auc'])

In [12]:
# cria dicionario de modelo
tabela_resultados ={}

In [13]:
def resumir_metricas(resultado):
    resumo = {
    'validacao accuracy': resultado['test_accuracy'].mean(),
    'validacao recall': resultado['test_recall'].mean(),
    'validacao f1': resultado['test_f1'].mean(),
    'validacao roc_auc': resultado['test_roc_auc'].mean(),
    'treino accuracy': resultado['train_accuracy'].mean(),
    'treino f1': resultado['train_f1'].mean(),
    'treino recall': resultado['train_recall'].mean(),
    'treino roc_auc': resultado['train_roc_auc'].mean(),
    }
    return resumo

In [14]:
tabela_resultados['Logistic_raw'] = resumir_metricas(resultado_raw)
display(tabela_resultados['Logistic_raw'])

{'validacao accuracy': np.float64(0.9671278026204696),
 'validacao recall': np.float64(0.9896588353369523),
 'validacao f1': np.float64(0.9823292922967923),
 'validacao roc_auc': np.float64(0.985711648818938),
 'treino accuracy': np.float64(0.9674795722867016),
 'treino f1': np.float64(0.9825169877399311),
 'treino recall': np.float64(0.990042211621172),
 'treino roc_auc': np.float64(0.9873569438581902)}

In [16]:
# chamado do CV(crossvalidadtion)
resultado_classweight = cross_validate(
    estimator = pipe_classweight,      # qual Pipeline avaliar
    X = X_train,              # as features de treino
    y = y_train,              # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['Logistic_classweight'] = resumir_metricas(resultado_classweight)
display(tabela_resultados['Logistic_classweight'])

{'validacao accuracy': np.float64(0.9586403577347633),
 'validacao recall': np.float64(0.957102774298001),
 'validacao f1': np.float64(0.977109542215153),
 'validacao roc_auc': np.float64(0.9858057743280941),
 'treino accuracy': np.float64(0.9598795104531493),
 'treino f1': np.float64(0.9777914917087329),
 'treino recall': np.float64(0.9570088439903234),
 'treino roc_auc': np.float64(0.9879354535957763)}

In [17]:
# chamado do CV(crossvalidadtion)
resultado_smote = cross_validate(
    estimator = pipe_smote,      # qual Pipeline avaliar
    X = X_train,              # as features de treino
    y = y_train,              # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['Logistic_smote'] = resumir_metricas(resultado_smote)
display(tabela_resultados['Logistic_smote'])

{'validacao accuracy': np.float64(0.960055036117452),
 'validacao recall': np.float64(0.9594001597034497),
 'validacao f1': np.float64(0.9779309680259939),
 'validacao roc_auc': np.float64(0.985619939832344),
 'treino accuracy': np.float64(0.9635910850383104),
 'treino f1': np.float64(0.9798882515214166),
 'treino recall': np.float64(0.9611260589587127),
 'treino roc_auc': np.float64(0.9876191993246352)}

### 4.1 Baseline: LogisticRegression em raw, class_weight e SMOTE

Comparação dos modelos utilizados para baseline de LogisticRegression

Raw:
- accuracy: 0.9671
- recall:0.9896
- f1:0.9823
- roc_auc: 0.9857

class_weight:
- accuracy: 0.9586
- recall: 0.9571
- f1: 0.9771
- roc_auc: 0.9858

Smote:
- accuracy: 0.9600
- recall: 0.9594
- f1: 0.9779
- roc_auc: 0.9856

### Leitura dos modelo

Paramêtros muito similares dos tres modelos de baseline analisados, mudanças muito sutis, porem o que chama a atenção é que o modelo raw, sem nenhum modificação de balanceamento apresentou melhor recall e f1-score.

###  Implicações

No dataset, hipertireoidismo é a classe positiva e majoritária(92%). A intuição inicial seria priorizar o recall da classe positiva, já que custo clínico maior costuma ser o falso negativo, deixar passar um doente. Neste dataset, porém, essa intuição engana: sendo a positiva a classe majoritária, seu recall satura facilmente (0.99 quase de graça) e não distingue os modelos. O erro difícil e clinicamente relevante aqui é classificar mal a classe negativa, a minoritária, onde o modelo tem mais dificuldade e onde os erros custam mais.
O class_weight e o SMOTE deslocam a atenção do modelo para a classe minoritária, a negativa. Ao fazer isso, o modelo passa a errar mais positivos para acertar mais negativos. Resultado: o recall da classe positiva cai nos baselines que utilizaram este ajustes.
O ROC-AUC é a métrica que menos se deixa enganar pelo desbalanceamento: os três estão praticamente empatados em 0.985. Isso diz que a capacidade de separação das três versões é essencialmente a mesma.

### Construção de desempate

Recall de 0.99 na classe que é 92% dos dados é fácil, o modelo acerta quase tudo porque quase tudo é positivo. A pergunta difícil, que nenhuma dessas métricas de validação está te mostrando isoladamente, é: quão bem o modelo pega a classe negativa, a minoritária?
 - abrir a classe negativa é o que vai diferenciar os modelos empatados.

### Métodos de analise da classe minoritaria

ferramenta é o classification_report ou a matriz de confusão, que mostram precisão e recall por classe separadamente.

In [22]:
# comparação dos modelos entre classes
# previsoes geradas por cross validation, uma por amostra
resultado_raw = cross_val_predict(pipe_raw, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, resultado_raw))

              precision    recall  f1-score   support

           0       0.85      0.70      0.77       218
           1       0.98      0.99      0.98      2611

    accuracy                           0.97      2829
   macro avg       0.91      0.84      0.87      2829
weighted avg       0.97      0.97      0.97      2829



In [24]:
# comparação dos modelos entre classes
# previsoes geradas por cross validation, uma por amostra
resultado_classweight = cross_val_predict(pipe_classweight, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, resultado_classweight))

              precision    recall  f1-score   support

           0       0.66      0.98      0.78       218
           1       1.00      0.96      0.98      2611

    accuracy                           0.96      2829
   macro avg       0.83      0.97      0.88      2829
weighted avg       0.97      0.96      0.96      2829



In [25]:
# comparação dos modelos entre classes
# previsoes geradas por cross validation, uma por amostra
resultado_smote = cross_val_predict(pipe_smote, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, resultado_smote))

              precision    recall  f1-score   support

           0       0.67      0.97      0.79       218
           1       1.00      0.96      0.98      2611

    accuracy                           0.96      2829
   macro avg       0.83      0.96      0.88      2829
weighted avg       0.97      0.96      0.96      2829



### Baseline: LogisticRegression em raw, class_weight e SMOTE na classe minoritária
Comparativo dos modelos baselines nas classes minoritarias (classe 0)


|Metricas    | RAW: |Class_weight |SMOTE:|
|------------|------|-------------|------|
|- precision | 0.85 |     0.66    | 0.67 |
|- f1-score  | 0.77 |     0.78    | 0.79 |
|- recall    | 0.70 |     0.98    | 0.97 |

### Comparação entre os modelo

Aqui é possviel ver uma diferenção substancial no recall em o modelo raw e os com medidas de balancemento, evidenciando o que nao tinha ficado claro apenos olhando a classe positiva, o modelo raw teve menos acertos na classe minoritária ,apesar de um bom recall no grupo positivo, e como previsto os modelos com ajuste de balancemento que acabaram tem indices menores que o modelo raw, ajustaram melhor a classe 0 sendo no recall [0.98] o que utilizou o hiperparametro de class_weight e de [0.97] o que utilizou o SMOTE para balanceamento

### Decisao do baseline e criterio de selecao de modelos (classification_report, cross_val_predict)

Recall da classe 1 (doente) / Recall da classe 0 (sadio):

- Raw: 0.99 / 0.70
- Class_weight: 0.96 / 0.98
- SMOTE: 0.96 / 0.97

A analise por classe revelou o trade-off que as metricas globais escondiam. O modelo raw pega quase todos os doentes (recall classe 1 = 0.99), mas classifica ~30% dos sadios como doentes (recall classe 0 = 0.70). As correcoes de balanceamento invertem:
resgatam a classe sadia (recall classe 0 sobe para 0.97/0.98), mas ao custo de deixar mais doentes passarem (recall classe 1 cai para 0.96).

#### Por que o balanceamento nao ajuda neste caso

A recomendacao geral de corrigir desbalanceamento pressupoe que a classe rara e a que importa proteger. Aqui a classe rara e o sadio, mas o erro clinicamente grave esta na classe majoritaria (o doente). Proteger a classe rara vai contra o objetivo clinico:
o balanceamento resolve um problema que este projeto nao tem e agrava o que tem.

#### Criterio de selecao (definido antes de avaliar os proximos modelos)

O briefing determina que nenhum caso de hipertireoidismo pode passar despercebido. Traduzido em metrica, o objetivo e maximizar o recall da classe 1 (doente).

Regra de decisao:

1. Criterio primario: maior recall da classe 1.
2. Criterio de desempate: se a diferenca no recall da classe 1 entre modelos for
   igual ou menor que 1 ponto percentual, considera-se empate tecnico, e o desempate passa para o maior recall da classe 0.

A margem de 1 ponto foi fixada antes de observar os resultados dos modelos de arvore,para evitar ajustar o criterio em favor de um resultado. Um ponto percentual equivale a aproximadamente 26 pacientes na classe doente, coerente com a severidade do criterio.

#### Baseline selecionado

LogisticRegression raw. Vence pelo criterio primario (recall classe 1 = 0.99, superior aos ramos balanceados por mais de 1 ponto). Custo assumido e documentado: recall da classe 0 de 0.70, ou seja, taxa elevada de falso alarme na classe sadia, aceita em troca de nao deixar doentes passarem.

#### Leitura alternativa (registrada para transparencia)

Uma metrica balanceada (recall macro) penalizaria o desequilibrio do raw (macro 0.845 contra ~0.97 dos balanceados). Optou-se por priorizar o criterio clinico do briefing sobre o equilibrio estatistico, de forma consciente. O ajuste fino do trade-off entre as classes sera tratado via limiar de decisao no notebook 04.

In [26]:
# preenchimento dos modelos de randomforest
pipe_rfraw = contrucao_pipeline(modelo=RandomForestClassifier(random_state=42), usa_smote=False)
pipe_rfclassweight = contrucao_pipeline(modelo=RandomForestClassifier(class_weight='balanced',random_state=42), usa_smote=False)
pipe_rfsmote = contrucao_pipeline(modelo=RandomForestClassifier(random_state=42), usa_smote=True)

In [27]:
# chamado do CV(crossvalidadtion)
resultado_rfraw = cross_validate(
    estimator = pipe_rfraw,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['rf_raw'] = resumir_metricas(resultado_rfraw)
display(tabela_resultados['rf_raw'])

# comparação dos modelos entre classes
pred_rfraw = cross_val_predict(pipe_rfraw, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, pred_rfraw))

{'validacao accuracy': np.float64(0.9791456893586415),
 'validacao recall': np.float64(0.9842992461704139),
 'validacao f1': np.float64(0.9886483724900759),
 'validacao roc_auc': np.float64(0.9933511122175773),
 'treino accuracy': np.float64(1.0),
 'treino f1': np.float64(1.0),
 'treino recall': np.float64(1.0),
 'treino roc_auc': np.float64(1.0)}

              precision    recall  f1-score   support

           0       0.83      0.92      0.87       218
           1       0.99      0.98      0.99      2611

    accuracy                           0.98      2829
   macro avg       0.91      0.95      0.93      2829
weighted avg       0.98      0.98      0.98      2829



### RandomForest raw

Recall classe 1 (doente): 0.98 | Recall classe 0 (sadio): 0.92

Pelo criterio de selecao: recall da classe 1 empata com o baseline dentro da margem de 1 ponto (0.99 vs 0.98), acionando o desempate pela classe 0, onde o RandomForest vence com folga (0.92 vs 0.70). Supera o baseline: mantem quase o mesmo recall de doentes com muito menos falso alarme na classe sadia.

**Alerta:** metricas de treino todas em 1.0 contra ~0.98 na validacao, indicando
overfitting (arvores crescem ate folhas puras com os parametros default). A resolver via limitacao de profundidade (max_depth, min_samples_leaf) na fase de tuning apenas no modelo final.

In [28]:
# chamado do CV(crossvalidadtion)
resultado_rfclass = cross_validate(
    estimator = pipe_rfclassweight,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['rf_classweight'] = resumir_metricas(resultado_rfclass)
display(tabela_resultados['rf_classweight'])

# comparação dos modelos entre classes
pred_rfclass = cross_val_predict(pipe_rfclassweight, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, pred_rfclass))

{'validacao accuracy': np.float64(0.9802045092091685),
 'validacao recall': np.float64(0.987362915100767),
 'validacao f1': np.float64(0.9892490695297053),
 'validacao roc_auc': np.float64(0.9942844169073162),
 'treino accuracy': np.float64(1.0),
 'treino f1': np.float64(1.0),
 'treino recall': np.float64(1.0),
 'treino roc_auc': np.float64(1.0)}

              precision    recall  f1-score   support

           0       0.86      0.89      0.87       218
           1       0.99      0.99      0.99      2611

    accuracy                           0.98      2829
   macro avg       0.92      0.94      0.93      2829
weighted avg       0.98      0.98      0.98      2829



### RandomForest class_weight

Recall classe 1 (doente): 0.99 | Recall classe 0 (sadio): 0.89

Pelo criterio de selecao: recall da classe 1 empata com o baseline dentro da margem de 1 ponto (0.99 vs 0.99), acionando o desempate pela classe 0, onde o RandomForest vence com folga novamente (0.89 vs 0.70). Supera o baseline: mantem o mesmo recall de doentes com muito menos falso alarme na classe sadia.


Tambem em comparação com o modelo raw, fica dentro do criterio de desempate de (1 ponto) mas acaba perdendo no desempate, mas vale o exercicio de analisar que o class_weight tem recall de doente de 0.99, colado no baseline, o que é atraente sob o objetivo de não deixar doente passar. A diferença entre raw e class_weight é minúscula, mantendo os dois como fortes candidatos a modelo final

**Alerta:** metricas de treino todas em 1.0 contra ~0.98 na validacao, indicando
overfitting (arvores crescem ate folhas puras com os parametros default). A resolver via limitacao de profundidade (max_depth, min_samples_leaf) na fase de tuning apenas no modelo final.

In [29]:
# chamado do CV(crossvalidadtion)
resultado_rfsmote = cross_validate(
    estimator = pipe_rfsmote,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['rf_smote'] = resumir_metricas(resultado_rfsmote)
display(tabela_resultados['rf_smote'])

# comparação dos modelos entre classes
pred_smote = cross_val_predict(pipe_rfsmote, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, pred_smote))

{'validacao accuracy': np.float64(0.978437099346446),
 'validacao recall': np.float64(0.9804678285458927),
 'validacao f1': np.float64(0.9882146481009411),
 'validacao roc_auc': np.float64(0.9923658418383235),
 'treino accuracy': np.float64(1.0),
 'treino f1': np.float64(1.0),
 'treino recall': np.float64(1.0),
 'treino roc_auc': np.float64(1.0)}

              precision    recall  f1-score   support

           0       0.80      0.95      0.87       218
           1       1.00      0.98      0.99      2611

    accuracy                           0.98      2829
   macro avg       0.90      0.97      0.93      2829
weighted avg       0.98      0.98      0.98      2829



### RandomForest SMOTE

Recall classe 1 (doente): 0.98 | Recall classe 0 (sadio): 0.95

Melhor variante do RandomForest pelo criterio de selecao.

Entre as tres variantes do RandomForest, o recall da classe 1 empata dentro da margem de 1 ponto (0.98 / 0.99 / 0.98), acionando o desempate pela classe 0. O SMOTE vence esse desempate (0.95), a frente do raw (0.92) e do class_weight (0.89): mantem quase todos os doentes e e o que menos gera falso alarme na classe sadia.

Contra o baseline (LogisticRegression raw): recall da classe 1 empata na margem
(0.98 vs 0.99), e no desempate pela classe 0 o RandomForest SMOTE vence com folga (0.95 vs 0.70). Supera o baseline.

Lider atual do comparativo de modelos.

**Alerta:** metricas de treino em 1.0 contra ~0.98 na validacao, overfitting estrutural das arvores com parametros default. Presente em todas as variantes do RandomForest, portanto nao afeta a comparacao entre elas. A tratar via tuning (max_depth,min_samples_leaf) apenas no modelo final escolhido.

In [30]:
# preenchimento dos modelos de randomforest
pipe_xgbraw = contrucao_pipeline(modelo=XGBClassifier(random_state=42), usa_smote=False)
pipe_xgbsmote = contrucao_pipeline(modelo=XGBClassifier(random_state=42), usa_smote=True)

In [36]:
# calcula scale_pos_weight a partir da proporcao real das classes para definir o pipeline utiliza o hiperparametro de class_weight dentro do XGBoost (scale_pos_weight)
num_negativos = (y_train == 0).sum() # contagem de numeros da classe 0
num_positivos = (y_train == 1).sum() # contagem de numeros da clases 1
peso = num_negativos / num_positivos #calcual o peso para o scale_pos_weight

print(f'negativos : {num_negativos}')
print(f'positivos : {num_positivos}')
print(f'peso : {peso}')

negativos : 218
positivos : 2611
peso : 0.0834929145921103


In [37]:
pipe_xgbweight = contrucao_pipeline(modelo=XGBClassifier(scale_pos_weight=peso,random_state=42), usa_smote=False)

In [34]:
# chamado do CV(crossvalidadtion)
resultado_xbgraw = cross_validate(
    estimator = pipe_xgbraw,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['XGBOOST_raw'] = resumir_metricas(resultado_xbgraw)
display(tabela_resultados['XGBOOST_raw'])

# comparação dos modelos entre classes
pred_xgbraw = cross_val_predict(pipe_xgbraw, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, pred_xgbraw))

{'validacao accuracy': np.float64(0.9738409581287719),
 'validacao recall': np.float64(0.9839146392386982),
 'validacao f1': np.float64(0.9858004998159451),
 'validacao roc_auc': np.float64(0.9818334739415697),
 'treino accuracy': np.float64(1.0),
 'treino f1': np.float64(1.0),
 'treino recall': np.float64(1.0),
 'treino roc_auc': np.float64(1.0)}

              precision    recall  f1-score   support

           0       0.82      0.85      0.83       218
           1       0.99      0.98      0.99      2611

    accuracy                           0.97      2829
   macro avg       0.90      0.92      0.91      2829
weighted avg       0.97      0.97      0.97      2829



### XGBoost raw

Recall classe 1 (doente): 0.98 | Recall classe 0 (sadio): 0.85

Contra o lider atual (RandomForest SMOTE, 0.98 / 0.95): recall da classe 1 empata exato (0.98), e no desempate pela classe 0 o XGBoost raw perde (0.85 vs 0.95).

 Fica abaixo do lider.

Comparado as variantes do RandomForest, o XGBoost raw tem o menor recall de classe 0 do grupo de arvores ate aqui. Nao supera o lider.

**Alerta:** treino em 1.0 contra ~0.97 na validacao, overfitting estrutural das
arvores, mesmo padrao do RandomForest. A tratar via tuning apenas no modelo final.

In [38]:
# chamado do CV(crossvalidadtion)
resultado_xbgweight = cross_validate(
    estimator = pipe_xgbweight,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['XGBOOST_weight'] = resumir_metricas(resultado_xbgweight)
display(tabela_resultados['XGBOOST_weight'])

# comparação dos modelos entre classes
pred_xgbweight = cross_val_predict(pipe_xgbweight, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, pred_xgbweight))

{'validacao accuracy': np.float64(0.9741949404296569),
 'validacao recall': np.float64(0.9747236324476386),
 'validacao f1': np.float64(0.9858438908304853),
 'validacao roc_auc': np.float64(0.9850593023387463),
 'treino accuracy': np.float64(0.9795865739996159),
 'treino f1': np.float64(0.9888170141874981),
 'treino recall': np.float64(0.9778822293018162),
 'treino roc_auc': np.float64(0.9982387141130478)}

              precision    recall  f1-score   support

           0       0.76      0.97      0.85       218
           1       1.00      0.97      0.99      2611

    accuracy                           0.97      2829
   macro avg       0.88      0.97      0.92      2829
weighted avg       0.98      0.97      0.98      2829



### XGBoost class_weight (scale_pos_weight)

Recall classe 1 (doente): 0.97 | Recall classe 0 (sadio): 0.97

Configurado com scale_pos_weight = 0.0835 (218 negativos / 2611 positivos), valor menor que 1 porque a classe rara neste dataset é a negativa. O parametro reponderou a funçao de perda para dar mais atencao a classe sadia.

Dois achados relevantes:

1. Equilibrio entre classes: recall de 0.97 nas duas classes, o mais equilibrado
   entre os modelos de arvore ate aqui. Empata em doente dentro da margem de 1 ponto e mantem recall alto na classe sadia.

2. Ausencia de overfitting: unico modelo de arvore ate agora cujo treino nao saturou em 1.0 (treino ~0.98 contra validacao ~0.97). O scale_pos_weight alterou o crescimento das arvores e evitou a memorizacao do treino que afeta as demais variantes de arvore.

Forte candidato a modelo final, pela saude estrutural (nao overfita). Posiçao definitiva no ranking sera confirmada na tabela consolidada com todos os modelos.

In [39]:
# chamado do CV(crossvalidadtion)
resultado_xbgsmote = cross_validate(
    estimator = pipe_xgbsmote,      # qual Pipeline avaliar
    X = X_train,               # as features de treino
    y = y_train,               # o alvo de treino
    cv = stratify,             # o objeto que corta os folds
    scoring = ['accuracy', 'f1', 'recall', 'roc_auc'],        # lista de metricas, como strings
    return_train_score = True
)

tabela_resultados['XGBOOST_smote'] = resumir_metricas(resultado_xbgsmote)
display(tabela_resultados['XGBOOST_smote'])

# comparação dos modelos entre classes
pred_xgbsmote = cross_val_predict(pipe_xgbsmote, X_train, y_train, cv=stratify)
# relatorio com precisao e recall separados por classe
print(classification_report(y_train, pred_xgbsmote))

{'validacao accuracy': np.float64(0.9777322617968041),
 'validacao recall': np.float64(0.9800854193680724),
 'validacao f1': np.float64(0.9878268071918933),
 'validacao roc_auc': np.float64(0.9870867914851486),
 'treino accuracy': np.float64(1.0),
 'treino f1': np.float64(1.0),
 'treino recall': np.float64(1.0),
 'treino roc_auc': np.float64(1.0)}

              precision    recall  f1-score   support

           0       0.80      0.95      0.87       218
           1       1.00      0.98      0.99      2611

    accuracy                           0.98      2829
   macro avg       0.90      0.96      0.93      2829
weighted avg       0.98      0.98      0.98      2829



## 4.3 Decisao de selecao do modelo final

### Consolidacao dos nove modelos

Recall por classe (validacao cruzada, cross_val_predict) e presenca de overfitting
(treino saturado em 1.0):

| Modelo | Recall classe 1 (doente) | Recall classe 0 (sadio) | Overfit |
|--------|--------------------------|-------------------------|---------|
| LogisticRegression raw | 0.99 | 0.70 | nao |
| LogisticRegression class_weight | 0.96 | 0.98 | nao |
| LogisticRegression SMOTE | 0.96 | 0.97 | nao |
| RandomForest raw | 0.98 | 0.92 | sim |
| RandomForest class_weight | 0.99 | 0.89 | sim |
| RandomForest SMOTE | 0.98 | 0.95 | sim |
| XGBoost raw | 0.98 | 0.85 | sim |
| XGBoost class_weight | 0.97 | 0.97 | nao |
| XGBoost SMOTE | 0.98 | 0.95 | sim |

### Filtro de validade (overfitting)

Antes de aplicar os criterios de recall, os modelos passam por um filtro de validade:
um modelo cujo desempenho de treino satura em 1.0 enquanto a validaçao fica abaixo tem generalizacao nao confirmada, e suas metricas de validaçao sao otimistas. Esse filtro nao é um criterio de preferencia, é um pre-requisito de confiabilidade,fundamentado em principio estatistico geral, nao no resultado especifico deste projeto.

Transparencia metodologica: o filtro de overfitting nao fazia parte do criterio inicial. Ele emergiu durante a experimentaçao, ao observar que todos os modelos de arvore, excetoo XGBoost class_weight, saturavam o treino. Foi incorporado como filtro de validade por principio, nao para favorecer nenhum modelo. A distincao adotada e qualitativa (treino satura em 1.0 versus nao satura), evitando um limiar numerico arbitrario.

Modelos reprovados no filtro (metricas nao confiaveis no estado bruto): RandomForest (raw, class_weight, SMOTE) e XGBoost (raw, SMOTE). Nao sao descartados por serem inferiores, e sim por terem metricas nao confiaveis enquanto overfitam. Poderiam retornar a disputa apos tuning que controlasse o overfit; opta-se, por escopo, por nao tuna-los, priorizando o candidato que ja e estruturalmente saudavel.

Modelos aprovados no filtro: LogisticRegression (raw, class_weight, SMOTE) e XGBoost class_weight.

### Aplicacao dos criterios aos modelos aprovados

Criterio primario (recall classe 1, topo = 0.99, margem de empate = 1 ponto):

- LogisticRegression raw: 0.99, dentro do topo, mas recall classe 0 = 0.70 (pior
  equilibrio, alto falso alarme na classe sadia).
- LogisticRegression class_weight e SMOTE: 0.96, fora da margem de 1 ponto do topo.
- XGBoost class_weight: 0.97, fora da margem de 1 ponto do topo por 1 ponto.

No estado bruto, entre os modelos saudaveis, nenhum combina recall classe 1 dentro da margem do topo com bom recall classe 0. O unico saudavel dentro do topo (LogisticRegression raw) tem o pior recall de classe 0. O candidato mais equilibrado (XGBoost class_weight,0.97/0.97) esta a 1 ponto de entrar na margem.

### Processo de decisao final

Para evitar afrouxar as regras em favor de um resultado desejado, as regras de recall permanecem fixas. O candidato saudavel mais promissor é promovido ao tuning e é reavaliado pelas mesmas regras. O alvo do tuning foi definido antes de sua execucao.

Candidato promovido ao tuning: XGBoost class_weight (scale_pos_weight = 0.0835).
Justificativa: unico modelo estruturalmente saudavel (sem overfit) com recall equilibrado entre as classes, a 1 ponto de entrar na margem do topo no criterio primario.

Alvo de sucesso do tuning (as tres condicoes, simultaneas):

1. Recall classe 1 (doente) >= 0.98. Ancorado na regra: dentro de 1 ponto do topo (0.99).
2. Recall classe 0 (sadio) >= 0.95. Ancorado no melhor recall classe 0 dos concorrentes do topo (RandomForest SMOTE e XGBoost SMOTE).
3. Sem overfit: treino nao pode saturar em 1.0; gap treino-validacao deve permanecer pequeno, como no estado atual (treino ~0.98, validacao ~0.97).

Regra de decisao:

- Se as tres condicoes forem satisfeitas: XGBoost class_weight tunado e o modelo final, avaliado no conjunto de teste no notebook 04.
- Se qualquer condicao falhar: o modelo final e o baseline (LogisticRegression raw), aceitando o custo documentado de recall classe 0 = 0.70.

 ---
**transparencia da escolha (criterio de possibilidade, nao de preferencia):**

O alvo foi fixado antes de qualquer execucao do tuning, tornando o processo imune a racionalizacao posterior.

Apenas um modelo foi levado ao tuning, para evitar a tendencia de tentativa e erro ate encontrar um resultado desejado. Esse modelo foi escolhido pelos criterios documentados neste notebook, por ser o melhor candidato a superar o baseline, ja que nenhum outro, dentro das dinamicas propostas, superou todas as regras no estado bruto.

O processo previa desfechos alternativos, definidos antes de conhecer os resultados:

- Se algum modelo tivesse superado todas as regras sem tuning, ele seguiria direto como modelo final para o notebook 04, sem esta etapa.
- Se nenhum modelo superasse as regras, o melhor candidato a bater o baseline seria levado ao tuning (o atual, ou outro que se mostrasse mais promissor), aplicando os mesmos criterios aqui definidos.

Ou seja, a escolha do XGBoost class_weight para o tuning decorre da posicao dele no criterio (unica possibilidade viavel de superar o baseline mantendo validade estrutural), nao de preferencia pelo modelo.


In [49]:
from sklearn.model_selection import RandomizedSearchCV

espaco_parametros = {
    'model__max_depth': [3, 4, 5, 6],
    'model__min_child_weight': [1, 3, 5],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__n_estimators': [50, 100, 200],
    'model__subsample': [0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.8, 0.9, 1.0],
    'model__scale_pos_weight': [0.05, peso, 0.2, 0.5]
}

In [50]:
# busca de parametro
busca_param = RandomizedSearchCV(
    estimator = pipe_xgbweight,
    param_distributions = espaco_parametros,
    n_iter = 60,
    scoring = 'recall_macro',
    cv = stratify,
    return_train_score = True,
    random_state = 42,
    n_jobs = -1
)
# realiza o fit
busca_param.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('columntrans',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('scaler',
                                                                               RobustScaler(),
                                                                               ['age',
                                                                                'TSH',
                                                                                'T3',
                                                                                'T4U',
                                                                                'FTI']),
                                                                              ('onehot',
                                                                               OneHotEncoder(handle_unknown='ignore',
                                                                                             sparse_output=False),
                                                                               ['referral '
                                                                                'source'])])),
                                             ('imputer',
                                              KNNImputer(w...
                   param_distributions={'model__colsample_bytree': [0.8, 0.9,
                                                                    1.0],
                                        'model__learning_rate': [0.01, 0.05,
                                                                 0.1],
                                        'model__max_depth': [3, 4, 5, 6],
                                        'model__min_child_weight': [1, 3, 5],
                                        'model__n_estimators': [50, 100, 200],
                                        'model__scale_pos_weight': [0.05,
                                                                    np.float64(0.0834929145921103),
                                                                    0.2, 0.5],
                                        'model__subsample': [0.8, 0.9, 1.0]},
                   random_state=42, return_train_score=True,
                   scoring='recall_macro')

In [51]:
# melhor combinacao de hiperparametros encontrada
print(busca_param.best_params_)
# melhor score de recall classe 1 (media dos folds)
print(busca_param.best_score_)

{'model__subsample': 0.8, 'model__scale_pos_weight': 0.05, 'model__n_estimators': 100, 'model__min_child_weight': 5, 'model__max_depth': 6, 'model__learning_rate': 0.1, 'model__colsample_bytree': 1.0}
0.9778679197611979


In [52]:
# previsoes do melhor modelo por cross validation
pred_tunado = cross_val_predict(busca_param.best_estimator_, X_train, y_train, cv=stratify)

# relatorio das duas classes
print(classification_report(y_train, pred_tunado))

              precision    recall  f1-score   support

           0       0.76      0.98      0.86       218
           1       1.00      0.97      0.99      2611

    accuracy                           0.97      2829
   macro avg       0.88      0.98      0.92      2829
weighted avg       0.98      0.97      0.98      2829



## 4.4 Tuning do XGBoost class_weight

### Primeira busca e achado do modelo degenerado

Primeira execucao do RandomizedSearchCV otimizando recall da classe 1 (criterio
primario). Resultado: best_score_ de 1.0 no recall da classe 1, com scale_pos_weight = 1.

A inspecao por classe revelou um modelo degenerado: recall classe 1 = 1.00 e recall classe 0 = 0.00. O modelo classificava todos os pacientes como doentes. Num dataset 92% positivo, o modo matematico de maximizar o recall da classe 1 e prever tudo como positivo, o que zera a classe sadia e torna o modelo clinicamente inutil.

Conclusao: otimizar recall da classe 1 puro e a metrica errada para o objetivo. O best_score_ elevado escondia um modelo sem valor. A escolha nao foi aceita.

### Correcao da metrica de otimizacao

O scoring da busca foi alterado para recall_macro (media do recall das duas classes), que penaliza o modelo degenerado (macro do degenerado = 0.50). O recall_macro nao substitui o criterio de selecao hierarquico; serve apenas para impedir que a busca convirja para solucoes degeneradas. O criterio final (recall classe 1 primario, classe 0 >= 0.95, sem overfit) continua sendo aplicado sobre os resultados.

### Estrategia de avaliacao

A busca reroda com multiplas metricas registradas (recall_classe1, recall_classe 0, recall_macro), refitando pela recall_macro mas guardando o recall por classe de todas as 60 combinacoes testadas.

Motivo: a busca escolhe o melhor pela recall_macro, mas a melhor combinacao pela macro pode nao ser a melhor pelo criterio hierarquico do projeto. Registrar o recall por classe de todas as combinacoes permite verificar, entre as 60 testadas, se alguma cumpre as tres condicoes do alvo, aplicando o criterio real do projeto e nao a metrica interna da busca.

Espaco de busca do scale_pos_weight: [0.05, 0.0835, 0.2, 0.5]. Demais hiperparametros controlam estrutura das arvores e regularizacao (max_depth, min_child_weight,learning_rate, n_estimators, subsample, colsample_bytree).

### Regra de decisao (reafirmada, definida antes dos resultados)

- Se alguma das 60 combinacoes cumprir as tres condicoes (recall classe 1 >= 0.98, recall classe 0 >= 0.95, sem overfit), a melhor delas e o modelo final.
- Se nenhuma cumprir, o modelo final e o baseline (LogisticRegression raw).
- O espaco de busca foi fixado a priori. Nao havera refinamento sucessivo da grade para forcar o alcance do alvo, o que caracterizaria ajuste ao resultado.

In [54]:
from sklearn.metrics import make_scorer, recall_score

# dicionario de metricas: recall de cada classe separado, mais o macro
metricas = {
    'recall_classe1': make_scorer(recall_score, pos_label=1),
    'recall_classe0': make_scorer(recall_score, pos_label=0),
    'recall_macro': 'recall_macro'
}

In [55]:
busca_param = RandomizedSearchCV(
    estimator = pipe_xgbweight,
    param_distributions = espaco_parametros,
    n_iter = 60,
    scoring = metricas,              # o dicionario, nao mais uma string
    refit = 'recall_macro',          # por qual metrica escolhe o best_
    cv = stratify,
    return_train_score = True,
    random_state = 42,
    n_jobs = -1
)

In [56]:
busca_param.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('columntrans',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('scaler',
                                                                               RobustScaler(),
                                                                               ['age',
                                                                                'TSH',
                                                                                'T3',
                                                                                'T4U',
                                                                                'FTI']),
                                                                              ('onehot',
                                                                               OneHotEncoder(handle_unknown='ignore',
                                                                                             sparse_output=False),
                                                                               ['referral '
                                                                                'source'])])),
                                             ('imputer',
                                              KNNImputer(w...
                                        'model__scale_pos_weight': [0.05,
                                                                    np.float64(0.0834929145921103),
                                                                    0.2, 0.5],
                                        'model__subsample': [0.8, 0.9, 1.0]},
                   random_state=42, refit='recall_macro',
                   return_train_score=True,
                   scoring={'recall_classe0': make_scorer(recall_score, response_method='predict', pos_label=0),
                            'recall_classe1': make_scorer(recall_score, response_method='predict', pos_label=1),
                            'recall_macro': 'recall_macro'})

In [58]:
# resultados das 60 combinacoes em tabela
resultados_busca = pd.DataFrame(busca_param.cv_results_)
# ver os nomes das colunas de metrica por classe
print([c for c in resultados_busca.columns if 'recall' in c])

['split0_test_recall_classe1', 'split1_test_recall_classe1', 'split2_test_recall_classe1', 'split3_test_recall_classe1', 'split4_test_recall_classe1', 'mean_test_recall_classe1', 'std_test_recall_classe1', 'rank_test_recall_classe1', 'split0_train_recall_classe1', 'split1_train_recall_classe1', 'split2_train_recall_classe1', 'split3_train_recall_classe1', 'split4_train_recall_classe1', 'mean_train_recall_classe1', 'std_train_recall_classe1', 'split0_test_recall_classe0', 'split1_test_recall_classe0', 'split2_test_recall_classe0', 'split3_test_recall_classe0', 'split4_test_recall_classe0', 'mean_test_recall_classe0', 'std_test_recall_classe0', 'rank_test_recall_classe0', 'split0_train_recall_classe0', 'split1_train_recall_classe0', 'split2_train_recall_classe0', 'split3_train_recall_classe0', 'split4_train_recall_classe0', 'mean_train_recall_classe0', 'std_train_recall_classe0', 'split0_test_recall_macro', 'split1_test_recall_macro', 'split2_test_recall_macro', 'split3_test_recall_macro

In [59]:
# filtro das condicoes sobre as 60 combinacoes
aprovadas = resultados_busca[
    (resultados_busca['mean_test_recall_classe1'] >= 0.98) &
    (resultados_busca['mean_test_recall_classe0'] >= 0.95)
]

# quantas passaram nas duas primeiras condicoes
print(len(aprovadas))

1


In [60]:
# ver os recalls e hiperparametros da combinacao aprovada
aprovadas[['mean_test_recall_classe1', 'mean_test_recall_classe0',
           'mean_train_recall_classe1', 'mean_train_recall_classe0', 'params']]

,mean_test_recall_classe1,mean_test_recall_classe0,mean_train_recall_classe1,mean_train_recall_classe0,params
58,0.981234,0.953911,0.983531,0.98509,"{'model__subsample': 0.9, 'model__scale_pos_we..."


In [61]:
# ver os hiperparametros completos da combinacao aprovada
aprovadas['params'].values

array([{'model__subsample': 0.9, 'model__scale_pos_weight': 0.5, 'model__n_estimators': 50, 'model__min_child_weight': 1, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}],
      dtype=object)

### Resultado do tuning e selecao do modelo final

A avaliacao das 60 combinacoes (Saida B) identificou uma configuracao que cumpre as tres condicoes do alvo definido antes do tuning:

| Condicao | Alvo | Combinacao 58 | Status |
|----------|------|---------------|--------|
| Recall classe 1 (doente) | >= 0.98 | 0.981 | passa |
| Recall classe 0 (sadio) | >= 0.95 | 0.954 | passa |
| Sem overfit | treino nao saturado | treino 0.98, gap < 5% | passa |

Nota sobre o criterio de overfit: adotou-se a nao saturacao do treino (treino nao atinge 1.0), criterio ja aplicado aos demais modelos do projeto, complementado pela verificacao de que o gap treino-validacao permanece na faixa que a literatura considera boa generalizacao (abaixo de 5%; classe 1 = 0.2%, classe 0 = 3.1%). Nao se adotou um limiar rigido de gap, que introduziria uma regua diferente da usada nos outros modelos e nao e sustentado pela literatura como linha de overfit.

Observacao metodologica: a combinacao escolhida pela busca (via recall_macro) tinha recall classe 1 de 0.97, falhando o alvo por 1 ponto. Foi a inspecao das 60 combinacoes pelo criterio real do projeto, e nao pela metrica interna da busca, que localizou a combinacao 58. Aceitar cegamente o best_estimator_ teria levado ao descarte indevido do XGBoost e a adocao do baseline.

### Modelo final selecionado

XGBoost tunado (combinacao 58). Hiperparametros:

- ubsample: 0.9
- scale_pos_weight: 0.5
- n_estimators: 50
- min_child_weight: 1
- max_depth: 4
- learning_rate: 0.05
- colsample_bytree: 1.0

Este modelo sera reconstruido e avaliado no conjunto de teste no notebook 04.

### Comparativo final: modelo escolhido vs baseline

| Modelo | Recall classe 1 | Recall classe 0 |
|--------|-----------------|-----------------|
| Baseline (LogisticRegression raw) | 0.99 | 0.70 |
| XGBoost tunado (final) | 0.98 | 0.95 |

O modelo final quase iguala o baseline no recall de doentes (diferenca de 1 ponto, dentro da margem definida) e o supera radicalmente no recall de sadios (0.95 vs 0.70), reduzindo drasticamente o falso alarme na classe sadia. Justifica-se a substituicao do baseline pelo modelo tunado, conforme a regra de decisao.

### Encaminhamento para o notebook 04

O notebook 04 ira:
1. Carregar o artefato (X_train, y_train, X_test, y_test).
2. Reconstruir o Pipeline com os hiperparametros da combinacao 58.
3. Treinar no X_train completo.
4. Avaliar uma unica vez no X_test, sem ajuste posterior.
5. Reportar as metricas finais (recall por classe, matriz de confusao, ROC-AUC).